# 01 — Ericsson KPI data processing and EDA

This notebook turns the **real Ericsson/AERPAW 5G NSA measurement files** into one clean table for the chatbot.

The raw release stores each KPI in a separate CSV (throughput, LTE/NR RSRP, LTE/NR SINR, CQI, MCS, RI and cell IDs). We **do not assume those rows line up**. We parse timestamps and join each KPI stream to the nearest throughput timestamp.

Output: `data/processed/kpi_observations.csv`.

Conceptually, this CSV is **the case we diagnose**. It is not the RAG knowledge base.

## 0. Environment

The repository targets **Python 3.12**. Install the exact notebook environment from the repo root:

```bash
python -m pip install -r requirements-dev.txt
```

The next cell makes `src/telecom_rag` importable whether Jupyter was started from the root or the `notebooks/` folder.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

## 1. Download the exact KPI dataset

The downloader is pinned to the **May 21, 2025** Dryad release and `Ericsson_Amir.zip`.

If Dryad blocks scripted downloading, the script tells you exactly where to place the manually downloaded ZIP; rerunning it then performs extraction and validation.

In [ ]:
import subprocess

subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "download_data.py")],
    check=False,
)

## 2. Discover the released KPI files

The experiment contains separate measurement streams and two UAV yaw-orientation groups. Before processing, inspect exactly what the downloader extracted.

In [ ]:
from telecom_rag.config import RAW_DATA_DIR
from telecom_rag.data import discover_metric_files

files = discover_metric_files(RAW_DATA_DIR)
for orientation, mapping in files.items():
    print(f"\n{orientation}")
    for metric, path in mapping.items():
        print(f"  {metric:20s} -> {path.relative_to(RAW_DATA_DIR)}")

## 3. Inspect one raw stream

Each author-provided CSV is normalized by `load_metric_file`: it standardizes column names, detects the timestamp, keeps location fields where present and identifies the KPI value column.

In [ ]:
from telecom_rag.data import load_metric_file

first_orientation = sorted(files)[0]
example_path = files[first_orientation]["throughput_mbps"]
example = load_metric_file(example_path, "throughput_mbps", numeric=True)
example.head()

## 4. Timestamp-align all KPIs

**Why not join by row number?** Independently logged telemetry can have different sampling rates or missing samples.

We therefore use throughput as the anchor (the performance outcome we later want to explain) and perform a nearest-time merge with a default **2 second tolerance**.

In [ ]:
from telecom_rag.data import build_kpi_table

kpis = build_kpi_table(RAW_DATA_DIR, merge_tolerance="2s")
print("Shape:", kpis.shape)
kpis.head()

## 5. Check data quality

Missing values after alignment are informative. They can mean a KPI was not reported sufficiently close to a throughput timestamp. Inspect missingness before making analytical claims.

In [ ]:
from telecom_rag.data import data_quality_report

quality = data_quality_report(kpis)
quality

In [ ]:
import matplotlib.pyplot as plt

missing = quality["missing_pct"].sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(missing.index, missing.values)
ax.set_xlabel("Missing values (%)")
ax.set_title("Missingness after timestamp alignment")
plt.tight_layout()
plt.show()

## 6. Add an unsupervised anomaly score

For the demo we want interesting observations without inventing rules such as “SINR below X is always bad.”

An **Isolation Forest** scores observations only by how unusual their KPI combination is relative to this dataset. The score is **not a fault label** and is not treated as ground truth.

In [ ]:
from telecom_rag.kpi import add_anomaly_scores

anomaly_result = add_anomaly_scores(kpis, contamination=0.05)
kpis = anomaly_result.dataframe
print("Features used:", anomaly_result.features)
print("Flagged observations:", int(kpis["anomaly_flag"].sum()))

## 7. Exploratory data analysis

This EDA is descriptive, not causal. We inspect ranges and relationships that can later provide **data-derived context** to the LLM.

In [ ]:
numeric_cols = [
    c for c in [
        "lte_rsrp_dbm", "nr_rsrp_dbm", "lte_sinr_db", "nr_sinr_db",
        "nr_cqi", "nr_mcs", "nr_ri", "throughput_mbps", "anomaly_score"
    ] if c in kpis.columns
]
kpis[numeric_cols].describe().T

In [ ]:
corr_cols = [c for c in numeric_cols if c != "anomaly_score"]
corr = kpis[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)), corr_cols, rotation=60, ha="right")
ax.set_yticks(range(len(corr_cols)), corr_cols)
ax.set_title("KPI correlation matrix")
fig.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

In [ ]:
if {"nr_sinr_db", "throughput_mbps"} <= set(kpis.columns):
    plot_df = kpis[["nr_sinr_db", "throughput_mbps"]].dropna()
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(plot_df["nr_sinr_db"], plot_df["throughput_mbps"], alpha=0.65)
    ax.set_xlabel("NR SINR (dB)")
    ax.set_ylabel("Throughput (Mbps)")
    ax.set_title("Measured throughput vs NR SINR")
    plt.tight_layout()
    plt.show()

## 8. Inspect candidate troubleshooting cases

The most unusual observations make useful demo cases. “Unusual” still does **not** mean “faulty”; it means they are worth investigating.

In [ ]:
display_cols = [
    c for c in [
        "observation_id", "timestamp", "orientation", "lte_rsrp_dbm", "nr_rsrp_dbm",
        "lte_sinr_db", "nr_sinr_db", "nr_cqi", "nr_mcs", "nr_ri",
        "throughput_mbps", "anomaly_score"
    ] if c in kpis.columns
]
kpis.sort_values("anomaly_score", ascending=False)[display_cols].head(15)

## 9. Save the processed observation table

This becomes the structured input to the final application:

- **KPI table** = measured network state being investigated.
- **RAG corpus** = technical knowledge used to interpret it.

In [ ]:
from telecom_rag.config import PROCESSED_KPI_PATH
from telecom_rag.data import save_kpi_table

saved = save_kpi_table(kpis, PROCESSED_KPI_PATH)
print(f"Saved {len(kpis):,} observations to {saved}")

## 10. Next step

We now have a reproducible data-science layer: real measurements, timestamp alignment, quality checks, EDA and anomaly scoring.

Notebook 02 builds the document corpus, embeddings, FAISS retrieval, LangGraph workflow and a controlled **same LLM: without RAG vs with RAG** evaluation.